In [ ]:
# === Inicialización de directorio de análisis organizado ===
from pathlib import Path
from datetime import datetime
import json, os

BASE_ANALISIS = Path('data/backtesting/ANALISIS')
BASE_ANALISIS.mkdir(parents=True, exist_ok=True)

# Generar timestamp base
_ts = datetime.now().strftime('%Y-%m-%d_%H-%M')
propuesto = BASE_ANALISIS / _ts
# Evitar colisión si se ejecuta varias veces el mismo minuto
if propuesto.exists():
    suf = 1
    while True:
        candidato = BASE_ANALISIS / f'{_ts}_{suf}'
        if not candidato.exists():
            propuesto = candidato
            break
        suf += 1
ANALISIS_RUN_DIR = propuesto
ANALISIS_RUN_DIR.mkdir(parents=True, exist_ok=False)

# Guardar metadatos
meta = {
    'created_at': datetime.now().isoformat(),
    'cwd': os.getcwd(),
    'description': 'Directorio raíz para esta sesión de análisis interactivo del notebook',
}
with open(ANALISIS_RUN_DIR / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print('Directorio de análisis activo:', ANALISIS_RUN_DIR)

In [1]:
import json

parametros_path = r"c:\Users\lenovo\bot_trading\semana_5\data\backtesting\parametros_seleccionados.json"

try:
    with open(parametros_path, "r", encoding="utf-8") as f:
        parametros = json.load(f)
    print("Parámetros actuales:")
    print(json.dumps(parametros, indent=4, ensure_ascii=False))
except FileNotFoundError:
    print(f"El archivo {parametros_path} no existe.")
except Exception as e:
    print(f"Error al leer el archivo de parámetros: {e}")

Parámetros actuales:
{
    "RSI_LIMIT_COMPRA": 45.0,
    "RSI_LIMIT_VENTA": 55.0,
    "ADX_LIMIT": 23.0,
    "MIN_VOTES": 4.0,
    "SL_MULT": 1.0,
    "TP_MULT": 2.0,
    "ATR_MIN": 0.5,
    "total_compras": 208.0,
    "total_ventas": 208.0,
    "ganancia_total": 61.31000000000017,
    "winrate": 48.07692307692308,
    "profit_factor": 1.5729371086814297,
    "max_drawdown": 15.840000000000828,
    "expectancy": 0.29475961538461626,
    "avg_duration": 1.5096153846153846
}


In [2]:
import pandas as pd
import glob
import gzip
import json
import os
from pathlib import Path

os.chdir(r"c:\Users\lenovo\bot_trading\semana_5")
print("Nuevo directorio de trabajo:", os.getcwd())

backtesting_path = "data/backtesting"
out_path = "out"

export_inspeccion = ANALISIS_RUN_DIR / 'inspeccion_inicial'
export_inspeccion.mkdir(parents=True, exist_ok=True)

def listar_archivos_recientes(path, extension="*"):
    archivos = glob.glob(f"{path}/**/*.{extension}", recursive=True)
    archivos = sorted(archivos, key=os.path.getmtime, reverse=True)
    return archivos

def previsualizar_jsonl_gz(file_path, n=5):
    try:
        with gzip.open(file_path, "rt", encoding="utf-8") as f:
            rows = []
            for i, line in enumerate(f):
                obj = json.loads(line)
                rows.append(obj)
                if i >= n - 1:
                    break
        with open(export_inspeccion / 'preview_decisiones.json', 'w', encoding='utf-8') as fw:
            json.dump(rows, fw, indent=2, ensure_ascii=False)
        print(f"Preview decisiones guardada en {export_inspeccion / 'preview_decisiones.json'}")
    except Exception as e:
        print(f"Error al leer {file_path}: {e}")

def analizar_csv(file_path):
    try:
        df = pd.read_csv(file_path)
        info = {
            'archivo': file_path,
            'columnas': df.columns.tolist(),
        }
        if "ganancia" in df.columns:
            ganancias = df["ganancia"].dropna()
            if not ganancias.empty:
                pos = ganancias[ganancias > 0].sum()
                neg = ganancias[ganancias <= 0].sum()
                pf = pos / abs(neg) if neg < 0 else None
                info.update({
                    'ganancia_total': float(ganancias.sum()),
                    'winrate_pct': float((ganancias > 0).mean() * 100),
                    'profit_factor': float(pf) if pf else None
                })
        with open(export_inspeccion / 'primer_csv_analizado.json', 'w', encoding='utf-8') as fw:
            json.dump(info, fw, indent=2, ensure_ascii=False)
        print('Resumen CSV guardado en', export_inspeccion / 'primer_csv_analizado.json')
    except Exception as e:
        print(f"Error al analizar {file_path}: {e}")

print("Archivos recientes en data/backtesting:")
archivos_csv = listar_archivos_recientes(backtesting_path, "csv")
print(archivos_csv[:5])

print("\nArchivos recientes en out/:")
archivos_jsonl_gz = listar_archivos_recientes(out_path, "jsonl.gz")
print(archivos_jsonl_gz[:5])

if archivos_jsonl_gz:
    previsualizar_jsonl_gz(archivos_jsonl_gz[0])

if archivos_csv:
    analizar_csv(archivos_csv[0])

Nuevo directorio de trabajo: c:\Users\lenovo\bot_trading\semana_5
Archivos recientes en data/backtesting:
['data/backtesting\\analisis backtesting 2025-09-08_13-30\\resumen_resultados_por_simbolo_periodo.csv', 'data/backtesting\\analisis backtesting 2025-09-06_23-23\\operaciones_robustas_avanzadas.csv', 'data/backtesting\\analisis backtesting 2025-09-06_23-23\\analisis_completo_por_simbolo_periodo_robusto.csv', 'data/backtesting\\analisis backtesting 2025-09-06_23-23\\operaciones_outliers_p1_p99.csv', 'data/backtesting\\analisis backtesting 2025-09-06_23-23\\top_20_perdidas.csv']

Archivos recientes en out/:
['out\\v3.2.1\\2025-09-07\\run_20250907_005325.jsonl.gz', 'out\\v3.2.1\\2025-09-05\\run_20250905_145418.jsonl.gz', 'out\\v3.2.1\\2025-09-04\\run_20250904_173418.jsonl.gz', 'out\\v3.2.1\\2025-09-04\\run_20250904_162500.jsonl.gz', 'out\\v3.2.1\\2025-09-04\\run_20250904_161217.jsonl.gz']

Previsualizando el primer archivo JSONL.GZ:

Analizando el primer archivo CSV:
Archivo: data/back

## Selección automática del último run y comparación entre runs
Este bloque detecta la carpeta más reciente de `data/backtesting` (formato YYYY-MM-DD_HH-MM), agrega métricas clave de cada run y construye un DataFrame comparativo para PF, winrate y drawdown.

In [3]:
import os, re, json, pandas as pd
from pathlib import Path

bt_root = Path('data/backtesting')
run_regex = re.compile(r'^\d{4}-\d{2}-\d{2}(_\d{2}-\d{2})$')

runs = []
for p in bt_root.iterdir():
    if p.is_dir() and run_regex.match(p.name):
        summary_file = p / 'summary.csv'
        if summary_file.exists():
            try:
                df_sum = pd.read_csv(summary_file)
                df_sum['run'] = p.name
                runs.append(df_sum)
            except Exception as e:
                print('Error leyendo', summary_file, e)

if not runs:
    print('No se encontraron runs con summary.csv')
else:
    df_runs = pd.concat(runs, ignore_index=True)
    cols_keep = [c for c in ['run','symbol','profit_factor','winrate','max_drawdown_pct','return_pct'] if c in df_runs.columns]
    comp = df_runs[cols_keep].copy()
    comp_sorted = comp.sort_values('run')
    ultimo_run = comp_sorted['run'].iloc[-1]
    print('Último run detectado:', ultimo_run)
    display(comp_sorted.tail(20))
    pivot_pf = comp.pivot_table(index='symbol', columns='run', values='profit_factor')
    pivot_wr = comp.pivot_table(index='symbol', columns='run', values='winrate')
    print('\nProfit Factor por símbolo/run:')
    display(pivot_pf)
    print('\nWinrate por símbolo/run:')
    display(pivot_wr)

    # Exportar artefactos dentro del directorio de análisis
    export_base = ANALISIS_RUN_DIR / 'comparacion_runs'
    export_base.mkdir(parents=True, exist_ok=True)
    comp_sorted.to_csv(export_base / 'runs_comparacion.csv', index=False)
    pivot_pf.to_csv(export_base / 'pivot_profit_factor.csv')
    pivot_wr.to_csv(export_base / 'pivot_winrate.csv')
    resumen_meta = {
        'ultimo_run': ultimo_run,
        'n_runs': comp_sorted['run'].nunique(),
    }
    with open(export_base / 'metadata.json', 'w', encoding='utf-8') as f:
        json.dump(resumen_meta, f, indent=2, ensure_ascii=False)
    print('Artefactos de comparación guardados en', export_base)

Último run detectado: 2025-09-07_00-53


,run,symbol,profit_factor,winrate,max_drawdown_pct,return_pct
0,2025-09-07_00-53,BTCUSDT,1.415318,58.823529,3.501777,54.62850
1,2025-09-07_00-53,ETHUSDT,1.732413,66.666667,0.809913,7.32120
2,2025-09-07_00-53,BNBUSDT,1.006786,53.191489,0.432203,0.01240
3,2025-09-07_00-53,WLDUSDT,1.341317,57.142857,0.001190,0.00114



Profit Factor por símbolo/run:


run,2025-09-07_00-53
symbol,
BNBUSDT,1.006786
BTCUSDT,1.415318
ETHUSDT,1.732413
WLDUSDT,1.341317



Winrate por símbolo/run:


run,2025-09-07_00-53
symbol,
BNBUSDT,53.191489
BTCUSDT,58.823529
ETHUSDT,66.666667
WLDUSDT,57.142857


## Test rápido thresholds alternativos (BNB, WLD)
Prueba variantes más permisivas: RSI compra < 40, ADX > 20, mantiene TP/SL actuales. Usa último run para ubicar datos de entrada si están disponibles; si no, busca en subcarpetas de símbolos.

## Backtest rápido con thresholds override (BNB, WLD)
Ejecuta el motor de backtesting con thresholds dinámicos sin modificar `config_estrategias.py`, compara contra baseline (run más reciente) y exporta artefactos CSV/Markdown en `data/backtesting/ANALISIS/threshold_tests/`.

In [5]:
import os, re, json, pandas as pd, numpy as np
from pathlib import Path
from datetime import datetime, timezone

# Importar motor y utilidades
import importlib.util, sys
core_path = Path('src/core/backtesting.py').resolve()
spec = importlib.util.spec_from_file_location('bt_module', core_path)
bt_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bt_module)  # type: ignore

try:
    from src.core.backtesting import cargar_y_combinar_datos, limpiar_ohlcv, backtesting
except Exception:
    cargar_y_combinar_datos = getattr(bt_module, 'cargar_y_combinar_datos')
    limpiar_ohlcv = getattr(bt_module, 'limpiar_ohlcv')
    backtesting = getattr(bt_module, 'backtesting')

class TempConfig:
    pass

baseline_metrics = {}
try:
    baseline_df = comp_sorted
    for _, row in baseline_df.iterrows():
        baseline_metrics[row.symbol] = {
            'profit_factor': row.profit_factor,
            'winrate': row.winrate,
            'max_dd': row.max_drawdown_pct,
        }
except Exception:
    pass

variants = [
    {"name": "base", "rsi_dynamic_buy": 45, "rsi_dynamic_sell": 55, "ADX_LIMIT": 23},
    {"name": "permisiva1", "rsi_dynamic_buy": 40, "rsi_dynamic_sell": 60, "ADX_LIMIT": 20},
    {"name": "permisiva2", "rsi_dynamic_buy": 38, "rsi_dynamic_sell": 62, "ADX_LIMIT": 20},
]

symbols_run = ["BNBUSDT", "WLDUSDT"]

master_rel = Path('src/core/..//data/historiales/historial_trading_limpio.csv').resolve()

results_rows = []
# Nuevo directorio de export dentro de la sesión de análisis
exports_dir = ANALISIS_RUN_DIR / 'threshold_tests'
exports_dir.mkdir(parents=True, exist_ok=True)

for sym in symbols_run:
    try:
        df_full = cargar_y_combinar_datos(str(master_rel), client=None, symbol=sym, meses=0)
    except Exception as e:
        print('Error cargando datos para', sym, e)
        continue
    if df_full is None or df_full.empty:
        print('Sin datos para', sym)
        continue
    df_full['timestamp'] = pd.to_datetime(df_full['timestamp'], errors='coerce', utc=True)
    end = df_full['timestamp'].max()
    cutoff = end - pd.Timedelta(days=30)
    df_30d = df_full[df_full['timestamp'] >= cutoff].copy()
    if df_30d.empty:
        print('Sin ventana 30d para', sym)
        continue
    df_30d = limpiar_ohlcv(df_30d)

    for var in variants:
        cfg = TempConfig()
        base_cfg = getattr(bt_module, 'config', None)
        fallback_vals = {
            'SL_MULT': getattr(base_cfg, 'SL_MULT', 1.0) if base_cfg else 1.0,
            'TP_MULT': getattr(base_cfg, 'TP_MULT', 2.0) if base_cfg else 2.0,
            'MIN_VOTES_COMPRA': getattr(base_cfg, 'MIN_VOTES_COMPRA', 1) if base_cfg else 1,
            'MIN_VOTES_VENTA': getattr(base_cfg, 'MIN_VOTES_VENTA', 1) if base_cfg else 1,
            'atr_min': getattr(base_cfg, 'atr_min', None) if base_cfg else None,
            'bb_width_min': getattr(base_cfg, 'bb_width_min', None) if base_cfg else None,
            'starting_capital': getattr(base_cfg, 'starting_capital', 100.0) if base_cfg else 100.0,
            'max_duracion': getattr(base_cfg, 'max_duracion', 999999) if base_cfg else 999999,
        }
        for k, v in fallback_vals.items():
            setattr(cfg, k, v)
        setattr(cfg, 'rsi_dynamic_buy', var['rsi_dynamic_buy'])
        setattr(cfg, 'rsi_dynamic_sell', var['rsi_dynamic_sell'])
        setattr(cfg, 'ADX_LIMIT', var['ADX_LIMIT'])
        setattr(cfg, 'RSI_LIMIT_COMPRA', var['rsi_dynamic_buy'])
        setattr(cfg, 'RSI_LIMIT_VENTA', var['rsi_dynamic_sell'])

        try:
            resultados, resumen = backtesting(df_30d.copy(), config_obj=cfg, writer=None)
        except Exception as e:
            print('Error backtesting', sym, var['name'], e)
            continue

        trades = sum(1 for r in resultados if r.get('tipo') == 'venta' and r.get('ganancia') is not None)
        row = {
            'symbol': sym,
            'variant': var['name'],
            'profit_factor': resumen.get('profit_factor'),
            'winrate': resumen.get('winrate'),
            'ganancia_total': resumen.get('ganancia_total'),
            'trades': trades,
            'max_drawdown': resumen.get('max_drawdown'),
            'signals_compra': resumen.get('signals_compra'),
            'signals_venta': resumen.get('signals_venta'),
        }
        base = baseline_metrics.get(sym, {})
        row['baseline_profit_factor'] = base.get('profit_factor')
        row['baseline_winrate'] = base.get('winrate')
        row['delta_pf'] = (row['profit_factor'] - row['baseline_profit_factor']) if base.get('profit_factor') is not None else None
        row['delta_trades'] = row['trades']
        results_rows.append(row)

# Consolidar y exportar
if results_rows:
    df_res = pd.DataFrame(results_rows)
    for sym in symbols_run:
        df_sym = df_res[df_res.symbol == sym]
        if not df_sym.empty:
            csv_path = exports_dir / f"{sym}_backtest_override.csv"
            df_sym.to_csv(csv_path, index=False)

    lines = ["# Threshold Override Tests", "", f"Export dir: {exports_dir}", ""]
    for sym in symbols_run:
        df_sym = df_res[df_res.symbol == sym]
        if df_sym.empty:
            continue
        lines.append(f"## {sym}")
        base_pf = baseline_metrics.get(sym, {}).get('profit_factor')
        lines.append(f"Baseline PF: {base_pf}")
        lines.append("")
        for _, r in df_sym.sort_values('variant').iterrows():
            lines.append(f"- {r.variant}: PF={r.profit_factor:.3f} (ΔPF={r.delta_pf if r.delta_pf is not None else 'NA'}) | Winrate={r.winrate:.2f}% | Trades={r.trades} | Signals C/V={r.signals_compra}/{r.signals_venta}")
        best = df_sym.loc[df_sym['profit_factor'].idxmax()]
        lines.append(f"**Mejor variante PF**: {best.variant} ({best.profit_factor:.3f})")
        lines.append("")
    summary_md = '\n'.join(lines)
    with open(exports_dir / 'summary.md', 'w', encoding='utf-8') as f:
        f.write(summary_md)
    df_res.to_csv(exports_dir / 'summary_variants.csv', index=False)
    print('\nArtefactos exportados en:', exports_dir)
    display(df_res)
else:
    print('Sin resultados (posible falta de datos o errores en backtesting).')

c:\Users\lenovo\bot_trading\semana_5\.venv\Lib\site-packages\pandas_ta\__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
2025-09-08 17:24:17,132 - INFO - Dataset 2881 filas tras filtro meses=0 (desde 2025-08-04 04:30:00+00:00 hasta 2025-09-03 04:30:00+00:00)
2025-09-08 17:24:17,136 - INFO - Dataset 2881 filas tras filtro meses=0 (desde 2025-08-04 04:30:00+00:00 hasta 2025-09-03 04:30:00+00:00)
2025-09-08 17:24:18,429 - INFO - RSI dinámico: compra=45, venta=55
2025-09-08 17:24:18,443 - INFO - SL/TP multipliers: k1=1.000, k2=2.000
2025-09-08 17:24:18,497 - INFO - Decisión registrada: {'evento': 'compra', 'precio': 752.5, 'timestamp': '2025-08-04T05:15:00+00:00', 'indicadores': {'RSI': 41.758241758240985, 'ATR': 1.48249

,symbol,variant,profit_factor,winrate,ganancia_total,trades,max_drawdown,signals_compra,signals_venta,baseline_profit_factor,baseline_winrate,delta_pf,delta_trades
0,BNBUSDT,base,1.906581,67.796610,118.880,59,30.750,584,38,1.006786,53.191489,0.899795,59
1,BNBUSDT,permisiva1,3.574825,70.769231,235.030,65,24.490,532,42,1.006786,53.191489,2.568038,65
2,BNBUSDT,permisiva2,2.353960,61.194030,170.450,67,23.780,453,40,1.006786,53.191489,1.347173,67
3,WLDUSDT,base,0.936170,45.945946,-0.024,37,0.125,0,37,1.341317,57.142857,-0.405147,37
4,WLDUSDT,permisiva1,1.147668,48.648649,0.057,37,0.152,0,38,1.341317,57.142857,-0.193649,37
5,WLDUSDT,permisiva2,1.203166,52.777778,0.077,36,0.127,0,37,1.341317,57.142857,-0.138151,36



Artefactos exportados en: data\backtesting\ANALISIS\threshold_tests\2025-09-08_20-24


In [ ]:
# === Modelo de costos: comisión y slippage ===
import pandas as pd, numpy as np, json, re
from pathlib import Path
from datetime import datetime

# Parámetros configurables del modelo de costos
COMISION_PCT = 0.001   # 0.1%
SLIPPAGE_PCT = 0.0005  # 0.05%

# Directorios origen
ultimo_run_dir = Path('data/backtesting/2025-09-07_00-53')  # usando último run con summary estable
thresholds_dir = Path('data/backtesting/ANALISIS/threshold_tests/2025-09-08_20-24')

# Directorio destino dentro de sesión actual de análisis
cost_dir = ANALISIS_RUN_DIR / 'modelo_costos'
cost_dir.mkdir(parents=True, exist_ok=True)

# Cargar baseline summary (multi-símbolo)
summary_path = ultimo_run_dir / 'summary.csv'
if not summary_path.exists():
    raise FileNotFoundError('No se encuentra summary.csv del último run')
summary_df = pd.read_csv(summary_path)

# Normalizar nombres esperados
# Esperamos columnas: symbol, profit_factor, winrate, ganancia_total (si existe), return_pct ...
# Si falta ganancia_total la derivamos desde equity final - inicial (si snapshot disponible)

# Helper para costo por trade: aplicamos comisión sobre notional de entrada y salida (2 * comision) + slippage en ambos lados
# Suponemos tamaño unitario relativo porque PnL ya está en USD neto bruto; restaremos costos absolutos derivando de ATR o de ganancia? =>
# En ausencia de notional por trade, aproximamos costo como porcentaje del valor absoluto del resultado + costo mínimo fijo relativo al capital base.
# Estrategia: si existe columna 'ganancia' por trade en archivos override, ajustamos: ganancia_net = ganancia - abs(ganancia)*total_pct_cost - base_capital*slippage_compensacion*(?)
# Simplicación razonable: costo = abs(ganancia) * (COMISION_PCT*2 + SLIPPAGE_PCT*2)
TOTAL_PCT_COST = COMISION_PCT*2 + SLIPPAGE_PCT*2  # ida y vuelta comisión + ida y vuelta slippage

# Función para recalcular métricas con costos
def recalcular_metricas(df_trades: pd.DataFrame):
    g = pd.to_numeric(df_trades['ganancia'], errors='coerce').dropna()
    if g.empty:
        return {'trades':0,'profit_factor_cost':np.nan,'winrate_cost':np.nan,'ganancia_total_cost':0.0}
    costos = g.abs() * TOTAL_PCT_COST
    g_net = g - costos
    pos = g_net[g_net > 0].sum()
    neg = g_net[g_net <= 0].sum()
    pf = (pos / abs(neg)) if neg < 0 else np.inf
    winrate = (g_net > 0).mean()*100
    return {
        'trades': int(len(g_net)),
        'profit_factor_cost': float(pf),
        'winrate_cost': float(winrate),
        'ganancia_total_cost': float(g_net.sum())
    }

# Cargar variantes BNB de threshold tests
variant_files = [p for p in thresholds_dir.glob('BNBUSDT_backtest_override.csv')]
variants_df = None
if variant_files:
    variants_df = pd.read_csv(variant_files[0])

# Recalcular variantes con costos (si tenemos per-variant granular?)
# El archivo override resume por variante; para aplicar costos necesitamos granularidad de trades original.
# Como sólo tenemos métricas agregadas, aproximaremos aplicando el factor de reducción estimado.
# Aproximación: Reducir ganancia_total por (1 - promedio(|g|*pct_cost)/|g|) = (1 - TOTAL_PCT_COST) si la magnitud media de ganancia ~ notional base.
# PF se ajusta recomputando suponiendo que tanto ganancias como pérdidas se reducen proporcionalmente.

def ajustar_resumen_variants(df_resumen: pd.DataFrame):
    adj_rows = []
    for _, r in df_resumen.iterrows():
        pf = r.get('profit_factor')
        win = r.get('winrate')
        trades = r.get('trades')
        total = r.get('ganancia_total')
        if pd.isna(pf) or total is None:
            continue
        # Ajuste proporcional
        total_cost = total * (1 - TOTAL_PCT_COST)
        # Para PF ajustado supondremos mismo ratio de reducción en ganancias y pérdidas => PF no cambia mucho salvo que pérdidas se vuelven ligeramente mayores por costo absoluto.
        # Implementamos una penalización simple: pf_cost = pf * (1 - TOTAL_PCT_COST*0.5)
        pf_cost = pf * (1 - TOTAL_PCT_COST*0.5)
        win_cost = win * (1 - TOTAL_PCT_COST*0.2)  # ligera degradación de winrate
        adj_rows.append({
            'symbol': r.symbol,
            'variant': r.variant,
            'profit_factor_orig': pf,
            'profit_factor_cost': pf_cost,
            'winrate_orig': win,
            'winrate_cost': win_cost,
            'trades': trades,
            'ganancia_total_orig': total,
            'ganancia_total_cost': total_cost
        })
    return pd.DataFrame(adj_rows)

adjusted_variants = ajustar_resumen_variants(variants_df) if variants_df is not None else None

# Baseline por símbolo: necesitamos trades detallados; usar archivos resultados_*_with_equity.csv dentro de cada símbolo
baseline_metrics_cost = []
for sym_dir in ['BTCUSDT','ETHUSDT','BNBUSDT','WLDUSDT']:
    sym_path = ultimo_run_dir / sym_dir
    if not sym_path.exists():
        continue
    trade_files = list(sym_path.glob('resultados_*_with_equity.csv'))
    if not trade_files:
        continue
    latest = max(trade_files, key=lambda p: p.stat().st_mtime)
    df_sym = pd.read_csv(latest)
    # Filtrar filas de cierre (ganancia no nula)
    df_trades = df_sym[pd.to_numeric(df_sym.get('ganancia'), errors='coerce').notna()].copy()
    met_cost = recalcular_metricas(df_trades)
    # baseline original del summary
    orig_row = summary_df[summary_df['symbol']==sym_dir].iloc[0] if not summary_df[summary_df['symbol']==sym_dir].empty else None
    baseline_metrics_cost.append({
        'symbol': sym_dir,
        'profit_factor_orig': float(orig_row['profit_factor']) if orig_row is not None and 'profit_factor' in orig_row else np.nan,
        'profit_factor_cost': met_cost['profit_factor_cost'],
        'winrate_orig': float(orig_row['winrate']) if orig_row is not None and 'winrate' in orig_row else np.nan,
        'winrate_cost': met_cost['winrate_cost'],
        'ganancia_total_cost': met_cost['ganancia_total_cost'],
        'trades': met_cost['trades']
    })

baseline_cost_df = pd.DataFrame(baseline_metrics_cost)

# Calcular deltas
if not baseline_cost_df.empty:
    baseline_cost_df['delta_pf'] = baseline_cost_df['profit_factor_cost'] - baseline_cost_df['profit_factor_orig']
    baseline_cost_df['delta_winrate'] = baseline_cost_df['winrate_cost'] - baseline_cost_df['winrate_orig']

# Export baseline cost metrics
baseline_cost_df.to_csv(cost_dir / 'metrics_cost_adjusted.csv', index=False)

# Export variants (BNB)
if adjusted_variants is not None and not adjusted_variants.empty:
    adjusted_variants['delta_pf'] = adjusted_variants['profit_factor_cost'] - adjusted_variants['profit_factor_orig']
    adjusted_variants['delta_winrate'] = adjusted_variants['winrate_cost'] - adjusted_variants['winrate_orig']
    adjusted_variants.to_csv(cost_dir / 'bnB_variants_cost_adjusted.csv', index=False)

# Markdown summary
def fmt(x):
    return 'NA' if pd.isna(x) else f"{x:.3f}" if isinstance(x, (int,float)) else str(x)

lines = [
    '# Modelo de Costos Aplicado',
    '',
    f'Comisión (round-trip aprox): {COMISION_PCT*2:.3%}',
    f'Slippage (round-trip aprox): {SLIPPAGE_PCT*2:.3%}',
    f'Factor total de reducción aplicado: {TOTAL_PCT_COST:.3%}',
    '',
    '## Baseline Ajustado',
]
for _, r in baseline_cost_df.iterrows():
    lines.append(f"- {r.symbol}: PF orig={fmt(r.profit_factor_orig)} -> PF cost={fmt(r.profit_factor_cost)} (Δ={fmt(r.delta_pf)}) | Winrate orig={fmt(r.winrate_orig)} -> {fmt(r.winrate_cost)} (Δ={fmt(r.delta_winrate)}) | Trades={r.trades}")

if adjusted_variants is not None and not adjusted_variants.empty:
    lines.extend(['','## BNB Variantes Ajustadas'])
    for _, r in adjusted_variants.iterrows():
        lines.append(f"- {r.variant}: PF orig={fmt(r.profit_factor_orig)} -> {fmt(r.profit_factor_cost)} (Δ={fmt(r.delta_pf)}) | Winrate orig={fmt(r.winrate_orig)} -> {fmt(r.winrate_cost)} (Δ={fmt(r.delta_winrate)}) | Ganancia total adj={fmt(r.ganancia_total_cost)}")

# Observaciones simples
lines.extend(['','## Observaciones','- Esta implementación aplica una aproximación proporcional al no disponer de notional individual por trade para variantes.','- Ajustar lógica cuando se integren tamaños de posición reales y fees exactas de exchange.','- Revisar si PF post-costos de BNB sigue por encima del umbral deseado para adoptar thresholds permisivos.'])

with open(cost_dir / 'summary_cost_model.md', 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

print('Exportados:')
print('-', cost_dir / 'metrics_cost_adjusted.csv')
if adjusted_variants is not None and not adjusted_variants.empty:
    print('-', cost_dir / 'bnB_variants_cost_adjusted.csv')
print('-', cost_dir / 'summary_cost_model.md')

display(baseline_cost_df)
if adjusted_variants is not None:
    display(adjusted_variants)

In [2]:
# Auto-generar estado_resumido al final de la sesión de análisis
import subprocess, sys, shutil, datetime as dt, pathlib

ANALISIS_DIR = pathlib.Path('../data/backtesting/ANALISIS').resolve()
# Heurística: elegir último run por orden cronológico que tenga summary.csv
BACKTESTING_DIR = pathlib.Path('../data/backtesting').resolve()
run_candidates = []
for p in BACKTESTING_DIR.iterdir():
    if p.is_dir() and (p / 'summary.csv').exists() and p.name[:4].isdigit():
        run_candidates.append(p)
run_dir = sorted(run_candidates)[-1] if run_candidates else None
print(f"Run base detectado: {run_dir}")

script_path = pathlib.Path('../scripts/generar_resumen_estado.py').resolve()
if not script_path.exists():
    raise FileNotFoundError(f"No se encontró el script generar_resumen_estado.py en {script_path}")

symbols = ['BTCUSDT','ETHUSDT','BNBUSDT','WLDUSDT']  # Ajustar si cambia universo
variants = ['1m','3m','6m','12m','30d']  # Ajustar si aplica
next_step = 'Refinar modelo de costos trade-a-trade y decidir adopción BNB permisiva'

if run_dir:
    cmd = [sys.executable, str(script_path), '--run-dir', str(run_dir.relative_to(BACKTESTING_DIR.parent)), '--analisis-dir', str(ANALISIS_DIR.relative_to(BACKTESTING_DIR.parent)), '--symbols', *symbols, '--variants', *variants, '--next-step', next_step]
    print('Ejecutando:', ' '.join(cmd))
    # Ejecutar dentro del root del proyecto
    proc = subprocess.run(cmd, cwd=BACKTESTING_DIR.parent, capture_output=True, text=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
        raise RuntimeError('Fallo generando estado_resumido')
else:
    print('No se detectó run con summary.csv; se omite generación de resumen.')

Run base detectado: C:\Users\lenovo\bot_trading\semana_5\data\backtesting\2025-09-07_00-53
Ejecutando: c:\Users\lenovo\bot_trading\semana_5\.venv\Scripts\python.exe C:\Users\lenovo\bot_trading\semana_5\scripts\generar_resumen_estado.py --run-dir backtesting\2025-09-07_00-53 --analisis-dir backtesting\ANALISIS --symbols BTCUSDT ETHUSDT BNBUSDT WLDUSDT --variants 1m 3m 6m 12m 30d --next-step Refinar modelo de costos trade-a-trade y decidir adopción BNB permisiva
Resumen actualizado: backtesting\ANALISIS\estado_resumido.md
Resumen archivado: backtesting\ANALISIS\RESÃšMENES\estado_resumido_2025-09-07_00-53.md

